In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from model.actor_critic import EncoderNet, MobileFrameObservationEncoderNet
from dataset import get_dataloader

from tqdm import trange
import tqdm

In [2]:
def cosine_loss(x, y):
    return 1. - F.cosine_similarity(x, y, dim=-1).mean()

def mse_loss(x, y):
    return F.mse_loss(x, y, reduction="mean")

In [3]:
class Alignment(nn.Module):
    def __init__(self, state_encoder, frame_encoder, state_feature_layer=-1):
        super().__init__()

        self.state_feature_layer = state_feature_layer

        self.state_encoder = state_encoder

        self.frame_encoder = frame_encoder

        self.state_encoder.eval()

        for param in self.state_encoder.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def encode_states(self, vectors):
        state_features = self.state_encoder.get_features(vectors)
        return state_features[self.state_feature_layer]

    def encode_frames(self, frames):
        frame_features = self.frame_encoder.get_features(frames)

        return frame_features
    
    def forward(self, frames, states):
        frame_features = self.encode_frames(frames)
        state_features = self.encode_states(states)

        return frame_features, state_features

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 10
encoder_weight, actor_weight, critic_wegit = torch.load("state_model.pth", weights_only=True)

state_encoder = EncoderNet(15, [512, 512, 512]).to(device)
frame_encoder = MobileFrameObservationEncoderNet(state_encoder.dim//2).to(device)

state_encoder.load_state_dict(encoder_weight)

model = Alignment(state_encoder, frame_encoder, -1).to(device)

optimizer = optim.Adam(model.frame_encoder.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=epochs)
dataloader = get_dataloader()
size = len(dataloader.dataset)
print(size)

Using cache found in /home/troja/.cache/torch/hub/pytorch_vision_main
/home/troja/miniconda3/envs/isaaclab/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/troja/miniconda3/envs/isaaclab/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


100000


In [5]:
for _ in trange(epochs, desc="Epochs"):
    running_loss = 0.0
    running_cosine_loss = 0.0
    running_mse_loss = 0.0
    for _, (vectors, frames) in enumerate(dataloader):
        vectors = vectors.to(device)
        frames = frames.to(device)
        
        frame_features, vector_features = model(frames, vectors)
        mse_losss_value = mse_loss(frame_features, vector_features)
        cosine_loss_value = cosine_loss(frame_features, vector_features)
        loss = 0.5 * mse_losss_value + 0.5 * cosine_loss_value
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * vectors.size(0)
        running_cosine_loss += cosine_loss_value.item() * vectors.size(0)
        running_mse_loss += mse_losss_value.item() * vectors.size(0)
 
    
    scheduler.step()
    train_loss = running_loss / size
    train_cosine_loss = running_cosine_loss / size
    train_mse_loss = running_mse_loss / size
    tqdm.tqdm.write(f"Train Loss: {train_loss:.4f}, Cosine Loss: {train_cosine_loss:.4f}, MSE Loss: {train_mse_loss:.4f},")

torch.save([model.frame_encoder.state_dict(), actor_weight, critic_wegit], "frame_model.pth")

Epochs:  10%|█         | 1/10 [01:34<14:13, 94.85s/it]

Train Loss: 0.1042, Cosine Loss: 0.0647, MSE Loss: 0.1437,


Epochs:  20%|██        | 2/10 [03:09<12:40, 95.00s/it]

Train Loss: 0.0459, Cosine Loss: 0.0281, MSE Loss: 0.0638,


Epochs:  30%|███       | 3/10 [04:42<10:57, 93.89s/it]

...

Train Loss: 0.0232, Cosine Loss: 0.0140, MSE Loss: 0.0324,


Epochs:  90%|█████████ | 9/10 [13:58<01:32, 92.64s/it]

Train Loss: 0.0205, Cosine Loss: 0.0123, MSE Loss: 0.0288,


Epochs: 100%|██████████| 10/10 [15:31<00:00, 93.12s/it]

Train Loss: 0.0189, Cosine Loss: 0.0113, MSE Loss: 0.0265,
